# USA Real Estate Market Analysis & Price Prediction

An end-to-end data science project analyzing 2.2 million US real estate listings.

## Project Overview
- **Dataset:** USA Real Estate Dataset from Kaggle (2.2M listings)
- **Goal:** Analyze housing market trends and predict house prices
- **Models:** Random Forest Regressor (R2: 0.859, MAE: $66,446)

## Table of Contents
1. Data Loading & Initial Exploration
2. Data Cleaning
3. Exploratory Data Analysis
4. Geographic Visualization
5. Price Prediction Model
6. Predict a House Price

## 1. Data Loading & Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import pgeocode
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv('realtor-data.zip.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
df.head()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

print(f'\nBasic statistics:')
df.describe()

## 2. Data Cleaning

In [ ]:
# Work on a copy to preserve original data
df2 = df.copy()

# Fix zip_code - convert float to proper 5-digit string with leading zeros
# e.g. 601.0 -> '00601'
df2['zip_code'] = df2['zip_code'].astype('string').str.split('.').str[0].str.zfill(5)

print('Sample zip codes after fix:')
print(df2['zip_code'].head(10))

In [ ]:
# Check nulls in critical columns
critical_cols = ['price', 'zip_code', 'house_size', 'bed', 'bath', 'city', 'state']
print('Nulls in critical columns:')
print(df2[critical_cols].isnull().sum())

print(f'\nTotal rows that would be dropped: {df2[critical_cols].isnull().any(axis=1).sum()}')

In [ ]:
# Drop rows with nulls in critical columns
df3 = df2.dropna(subset=critical_cols).reset_index(drop=True)
print(f'Shape after dropping nulls: {df3.shape}')

In [ ]:
# Remove price outliers using IQR method per state
# This ensures expensive California homes aren't flagged as outliers
# compared to cheaper states
def remove_outliers_iqr(group):
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return group[(group['price'] >= lower) & (group['price'] <= upper)]

df4 = df3.groupby('state', group_keys=False).apply(remove_outliers_iqr, include_groups=False)

# Also remove unrealistically low prices (below $150k)
df4 = df4[df4['price'] >= 150000]

print(f'Shape after outlier removal: {df4.shape}')
print(f'\nPrice statistics after cleaning:')
print(df4['price'].describe().map('${:,.0f}'.format))

## 3. Exploratory Data Analysis

In [ ]:
# Average price by state
avg_by_state = df4.groupby('state')['price'].mean().sort_values(ascending=False)
print('Average price by state:')
print(avg_by_state.map('${:,.0f}'.format))

In [ ]:
# Price distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df4['price'].hist(bins=50, color='steelblue', edgecolor='white')
plt.title('Price Distribution')
plt.xlabel('Price ($)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
avg_by_state.head(15).plot(kind='bar', color='steelblue')
plt.title('Top 15 States by Average Price')
plt.xlabel('State')
plt.ylabel('Average Price ($)')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['price', 'bed', 'bath', 'acre_lot', 'house_size']
plt.figure(figsize=(8, 6))
sns.heatmap(df4[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Geographic Visualization

In [ ]:
# State coordinates for map
state_coords = {
    'Alabama': [32.806671, -86.791130], 'Alaska': [61.370716, -152.404419],
    'Arizona': [33.729759, -111.431221], 'Arkansas': [34.969704, -92.373123],
    'California': [36.116203, -119.681564], 'Colorado': [39.059811, -105.311104],
    'Connecticut': [41.597782, -72.755371], 'Delaware': [39.318523, -75.507141],
    'District of Columbia': [38.897438, -77.026817], 'Florida': [27.766279, -81.686783],
    'Georgia': [33.040619, -83.643074], 'Guam': [13.444304, 144.793731],
    'Hawaii': [21.094318, -157.498337], 'Idaho': [44.240459, -114.478828],
    'Illinois': [40.349457, -88.986137], 'Indiana': [39.849426, -86.258278],
    'Iowa': [42.011539, -93.210526], 'Kansas': [38.526600, -96.726486],
    'Kentucky': [37.668140, -84.670067], 'Louisiana': [31.169960, -91.867805],
    'Maine': [44.693947, -69.381927], 'Maryland': [39.063946, -76.802101],
    'Massachusetts': [42.230171, -71.530106], 'Michigan': [43.326618, -84.536095],
    'Minnesota': [45.694454, -93.900192], 'Mississippi': [32.741646, -89.678696],
    'Missouri': [38.456085, -92.288368], 'Montana': [46.921925, -110.454353],
    'Nebraska': [41.125370, -98.268082], 'Nevada': [38.313515, -117.055374],
    'New Hampshire': [43.452492, -71.563896], 'New Jersey': [40.298904, -74.521011],
    'New Mexico': [34.840515, -106.248482], 'New York': [42.165726, -74.948051],
    'North Carolina': [35.630066, -79.806419], 'North Dakota': [47.528912, -99.784012],
    'Ohio': [40.388783, -82.764915], 'Oklahoma': [35.565342, -96.928917],
    'Oregon': [44.572021, -122.070938], 'Pennsylvania': [40.590752, -77.209755],
    'Puerto Rico': [18.220833, -66.590149], 'Rhode Island': [41.680893, -71.511780],
    'South Carolina': [33.856892, -80.945007], 'South Dakota': [44.299782, -99.438828],
    'Tennessee': [35.747845, -86.692345], 'Texas': [31.054487, -97.563461],
    'Utah': [40.150032, -111.862434], 'Vermont': [44.045876, -72.710686],
    'Virgin Islands': [18.335765, -64.896335], 'Virginia': [37.769337, -78.169968],
    'Washington': [47.400902, -121.490494], 'West Virginia': [38.491226, -80.954453],
    'Wisconsin': [44.268543, -89.616508], 'Wyoming': [42.755966, -107.302490]
}

# Calculate state statistics
state_stats = df4.groupby('state')['price'].agg(['mean', 'median', 'min', 'max']).reset_index()
state_stats.columns = ['state', 'avg_price', 'median_price', 'min_price', 'max_price']
state_stats['lat'] = state_stats['state'].map(lambda x: state_coords.get(x, [None, None])[0])
state_stats['lon'] = state_stats['state'].map(lambda x: state_coords.get(x, [None, None])[1])
state_stats = state_stats.dropna(subset=['lat', 'lon'])

# Create state level map
m_state = folium.Map(location=[39.5, -98.35], zoom_start=4)

for _, row in state_stats.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=10,
        popup=f"""
        {row['state']}
        Avg: ${row['avg_price']:,.0f}
        Median: ${row['median_price']:,.0f}
        Min: ${row['min_price']:,.0f}
        Max: ${row['max_price']:,.0f}
        """,
        color='blue',
        fill=True
    ).add_to(m_state)

m_state.save('avg_price_by_state.html')
print('State map saved!')

In [ ]:
# Create zip code level map using pgeocode for coordinates
nomi = pgeocode.Nominatim('us')

zip_stats = df4.groupby('zip_code')['price'].agg(['mean', 'median', 'min', 'max']).reset_index()
zip_stats.columns = ['zip_code', 'avg_price', 'median_price', 'min_price', 'max_price']

coords = nomi.query_postal_code(zip_stats['zip_code'].tolist())
zip_stats['lat'] = coords['latitude'].values
zip_stats['lon'] = coords['longitude'].values
zip_stats = zip_stats.dropna(subset=['lat', 'lon'])

# Create zip code map with bubble size based on price
m_zip = folium.Map(location=[39.5, -98.35], zoom_start=4)

for _, row in zip_stats.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['avg_price'] / 100000,
        popup=f"""
        Zip: {row['zip_code']}
        Avg: ${row['avg_price']:,.0f}
        Median: ${row['median_price']:,.0f}
        Min: ${row['min_price']:,.0f}
        Max: ${row['max_price']:,.0f}
        """,
        color='blue',
        fill=True,
        fill_opacity=0.4
    ).add_to(m_zip)

m_zip.save('avg_price_by_zip.html')
print('Zip code map saved!')

## 5. Price Prediction Model

In [ ]:
# Prepare model dataframe
model_df = df4.copy()

# Label encode city and state (separate encoders so we can decode later)
le_city = LabelEncoder()
le_state = LabelEncoder()
model_df['city'] = le_city.fit_transform(model_df['city'])
model_df['state'] = le_state.fit_transform(model_df['state'])

# Convert zip_code to numeric
model_df['zip_code'] = pd.to_numeric(model_df['zip_code'], errors='coerce')
model_df = model_df.dropna(subset=['zip_code'])

# Define features and target
X = model_df[['bed', 'bath', 'house_size', 'acre_lot', 'zip_code', 'city', 'state']]
y = model_df['price']

print(f'Features shape: {X.shape}')
print(f'\nFeature types:\n{X.dtypes}')

In [ ]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

In [ ]:
# Find best parameters using GridSearchCV on a sample (to save time)
X_train_sample = X_train.sample(100000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_sample, y_train_sample)
print(f'Best params: {rf_grid.best_params_}')
print(f'Best R2 (sample): {rf_grid.best_score_:.3f}')

In [ ]:
# Train final model on full dataset with best parameters
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Evaluate model
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'=== Model Results ===')
print(f'MAE: ${mae:,.0f}')
print(f'R2 Score: {r2:.3f}')

In [ ]:
# Feature importance visualization
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Random Forest Feature Importance - House Price')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(importance_df)

## 6. Predict a House Price

Use the trained model to predict the price of any house in the dataset.

In [ ]:
def predict_price(bed, bath, house_size, acre_lot, zip_code, city, state):
    """
    Predict house price given property details.
    
    Parameters:
    - bed: number of bedrooms
    - bath: number of bathrooms  
    - house_size: living area in square feet
    - acre_lot: lot size in acres
    - zip_code: 5 digit zip code as string
    - city: city name (must exist in dataset)
    - state: state name (must exist in dataset)
    """
    city_encoded = le_city.transform([city])[0]
    state_encoded = le_state.transform([state])[0]
    zip_numeric = pd.to_numeric(zip_code)
    
    input_data = pd.DataFrame({
        'bed': [bed],
        'bath': [bath],
        'house_size': [house_size],
        'acre_lot': [acre_lot],
        'zip_code': [zip_numeric],
        'city': [city_encoded],
        'state': [state_encoded]
    })
    
    prediction = rf_model.predict(input_data)
    print(f'Predicted Price: ${prediction[0]:,.0f}')

# Example predictions
predict_price(
    bed=3, bath=2, house_size=1500, acre_lot=0.15,
    zip_code='90210', city='Beverly Hills', state='California'
)

predict_price(
    bed=3, bath=2, house_size=1500, acre_lot=0.15,
    zip_code='48201', city='Detroit', state='Michigan'
)